### Create a tf-idf-based classificator model for the bank77 dataset

In [1]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import GradientBoostingClassifier
from utils.utils import preprocessing
import shutup
shutup.please()

[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\vojta\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\vojta\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\vojta\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [3]:
import requests
import pandas as pd
from io import StringIO

# URLs for the files
urls = [
    "https://raw.githubusercontent.com/food-hazard-detection-semeval-2025/food-hazard-detection-semeval-2025.github.io/refs/heads/main/data/incidents_train.csv",
    "https://raw.githubusercontent.com/food-hazard-detection-semeval-2025/food-hazard-detection-semeval-2025.github.io/refs/heads/main/data/incidents_valid.csv",
    "https://raw.githubusercontent.com/food-hazard-detection-semeval-2025/food-hazard-detection-semeval-2025.github.io/refs/heads/main/data/incidents_test.csv"
]

# Load each file into a DataFrame
dataframes = []
for url in urls:
    response = requests.get(url)
    response.raise_for_status()  # Raise an error for bad status codes
    csv_data = StringIO(response.text)  # Convert text to a file-like object
    df_orig = pd.read_csv(
        csv_data,
        engine='python',             # supports multiline quoted fields
        quoting=0,                   # QUOTE_MINIMAL
        quotechar='"',
        on_bad_lines='warn'          # skip lines that still cause issues
    )
    dataframes.append(df_orig)

# Access the DataFrames
train_df, valid_df, test_df = dataframes
train_df = pd.concat([train_df, valid_df])

train_df['text'] = train_df['text'].apply(lambda x: preprocessing(x))
train_df['title'] = train_df['title'].apply(lambda x: preprocessing(x))
train_df['combined'] = train_df['title'] + ' ' +  train_df['text']


test_df['text'] = test_df['text'].apply(lambda x: preprocessing(x))
test_df['title'] = test_df['title'].apply(lambda x: preprocessing(x))
test_df['combined'] = test_df['title'] + ' ' +  test_df['text']


y_train = train_df['hazard-category']
hazard_true = test_df['hazard-category']

X_train = train_df['combined']
X_test = test_df['combined']

In [4]:
# # Create a tf-idf matrix
vectorizer = TfidfVectorizer()
X_train = vectorizer.fit_transform(X_train)
X_test = vectorizer.transform(X_test)

In [5]:
# 1) Libraries
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score


# ----------------------------------------------------------------
# 4) Definice modelu RandomForestClassifier
model = RandomForestClassifier(random_state=42)

# 5) Nastavení rozsahu parametrů pro RandomizedSearchCV
param_dist = {
    "n_estimators": [50, 100, 200],       # Počet stromů v lese
    "max_depth": [3, 5, 10, None],        # Maximální hloubka stromu
    "min_samples_split": [2, 5, 10],      # Minimální počet vzorků pro split
    "min_samples_leaf": [1, 2, 5],        # Minimální počet vzorků v listu
}

# 6) Konfigurace RandomizedSearchCV (n_iter a cv lze upravit dle potřeby)
random_search = RandomizedSearchCV(
    model,
    param_distributions=param_dist,
    n_iter=10,             # kolik náhodných kombinací parametrů prozkoumat
    cv=5,                  # 5-fold cross-validace
    scoring="f1_macro",    # metrika, dle které se bude model porovnávat
    random_state=42,
    n_jobs=-1,             # využití všech CPU jader pro rychlejší výpočet
    verbose=1
)

# 7) Trénink modelu s vyhledáváním nejlepších hyperparametrů
random_search.fit(X_train, y_train)

# 8) Vypsání nejlepších parametrů a skóre
print("Nejlepší parametry:", random_search.best_params_)
print("Nejlepší skóre na trénovací cross-validaci:", random_search.best_score_)

# 9) Ověření na testovací sadě
best_model = random_search.best_estimator_  # získáme nejlepší nalezený model
hazard_pred = best_model.predict(X_test)

# 10) Vyhodnocení
print("Přesnost na testu:", accuracy_score(hazard_true, hazard_pred))
print("Classification report na testu:")
print(classification_report(hazard_true ,hazard_pred, zero_division=0))


Fitting 5 folds for each of 10 candidates, totalling 50 fits
Nejlepší parametry: {'n_estimators': 200, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_depth': None}
Nejlepší skóre na trénovací cross-validaci: 0.5008507760870218
Přesnost na testu: 0.9087261785356068
Classification report na testu:
                                precision    recall  f1-score   support

                     allergens       0.94      0.99      0.96       365
                    biological       0.91      0.99      0.94       343
                      chemical       0.97      0.65      0.78        52
food additives and flavourings       1.00      0.50      0.67         4
                foreign bodies       0.82      0.99      0.90       111
                         fraud       0.94      0.64      0.76        75
                     migration       0.00      0.00      0.00         1
          organoleptic aspects       1.00      0.20      0.33        10
                  other hazard       0.75      0.

In [6]:
y_train = train_df['product-category']
product_true = test_df['product-category']


# 1) Libraries
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import classification_report, accuracy_score


# ----------------------------------------------------------------
# 4) Definice modelu RandomForestClassifier
model = RandomForestClassifier(random_state=42)

# 5) Nastavení rozsahu parametrů pro RandomizedSearchCV
param_dist = {
    "n_estimators": [100, 200, 300, 500],       # Počet stromů v lese
    "max_depth": [3, 5, 10, None],        # Maximální hloubka stromu
    "min_samples_split": [2, 5, 10],      # Minimální počet vzorků pro split
    "min_samples_leaf": [1, 2, 5],        # Minimální počet vzorků v listu
}

# 6) Konfigurace RandomizedSearchCV (n_iter a cv lze upravit dle potřeby)
random_search = RandomizedSearchCV(
    model,
    param_distributions=param_dist,
    n_iter=10,             # kolik náhodných kombinací parametrů prozkoumat
    cv=5,                  # 5-fold cross-validace
    scoring="f1_macro",    # metrika, dle které se bude model porovnávat
    random_state=42,
    n_jobs=-1,             # využití všech CPU jader pro rychlejší výpočet
    verbose=1
)

# 7) Trénink modelu s vyhledáváním nejlepších hyperparametrů
random_search.fit(X_train, y_train)

# 8) Vypsání nejlepších parametrů a skóre
print("Nejlepší parametry:", random_search.best_params_)
print("Nejlepší skóre na trénovací cross-validaci:", random_search.best_score_)


# 9) Ověření na testovací sadě
best_model = random_search.best_estimator_  # získáme nejlepší nalezený model
product_pred = best_model.predict(X_test)

# 10) Vyhodnocení
print("Přesnost na testu:", accuracy_score(product_true, product_pred))
print("Classification report na testu:")
print(classification_report(product_true, product_pred, zero_division=0))


Fitting 5 folds for each of 10 candidates, totalling 50 fits
Nejlepší parametry: {'n_estimators': 200, 'min_samples_split': 10, 'min_samples_leaf': 1, 'max_depth': None}
Nejlepší skóre na trénovací cross-validaci: 0.30397308421856567
Přesnost na testu: 0.5927783350050151
Classification report na testu:
                                                   precision    recall  f1-score   support

                              alcoholic beverages       0.83      0.62      0.71        16
                      cereals and bakery products       0.40      0.76      0.53       121
     cocoa and cocoa preparations, coffee and tea       0.83      0.48      0.61        42
                                    confectionery       0.86      0.18      0.30        33
dietetic foods, food supplements, fortified foods       0.75      0.23      0.35        26
                                    fats and oils       1.00      0.33      0.50         6
                   food additives and flavourings       1.

### Dummy Classifier

In [11]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import recall_score, f1_score, precision_score, accuracy_score
dummy = DummyClassifier(strategy='most_frequent', random_state=42)
y_train = train_df['hazard-category']
dummy.fit(X_train, y_train)
y_pred = dummy.predict(X_test)
display(f"Dummy Accuracy score: {accuracy_score(hazard_true, y_pred)}")
display(f"Dummy Recall score: {recall_score(hazard_true, y_pred, average='macro')}")
display(f"Dummy F1 score: {f1_score(hazard_true, y_pred, average='macro')}")
display(f"Dummy Precision score: {precision_score(hazard_true, y_pred, average='macro', zero_division=0)}")


'Dummy Accuracy score: 0.36609829488465395'

'Dummy Recall score: 0.1'

'Dummy F1 score: 0.05359765051395007'

'Dummy Precision score: 0.0366098294884654'

In [12]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import recall_score, f1_score, precision_score, accuracy_score
dummy = DummyClassifier(strategy='most_frequent', random_state=42)
y_train = train_df['product-category']
dummy.fit(X_train, y_train)
y_pred = dummy.predict(X_test)
display(f"Dummy Accuracy score: {accuracy_score(product_true, y_pred)}")
display(f"Dummy Recall score: {recall_score(product_true, y_pred, average='macro')}")
display(f"Dummy F1 score: {f1_score(product_true, y_pred, average='macro')}")
display(f"Dummy Precision score: {precision_score(product_true, y_pred, average='macro', zero_division=0)}")


'Dummy Accuracy score: 0.28284854563691075'

'Dummy Recall score: 0.05'

'Dummy F1 score: 0.022048475371383894'

'Dummy Precision score: 0.014142427281845537'